# KolektorSDD2 — Phase 3 error analysis

Reads the report produced by `scripts/run_error_analysis.py`.
That script uses the frozen Phase 2 checkpoint and the val-chosen threshold.
It does not train and does not retune on test.

Problem statement: see `PROBLEM.md`.

In [ ]:
from pathlib import Path
import json
from IPython.display import Image, Markdown, display

ROOT = Path("..").resolve()
REPORT_DIR = ROOT / "reports" / "error_analysis"
report = json.loads((REPORT_DIR / "error_report.json").read_text(encoding="utf-8"))
print("Generated UTC:", report["generated_at_utc"])
print("Device:", report["device"])
print("Threshold:", report["threshold"])
print("Test confusion:", report["test_confusion"])

## Test false negatives

Primary costly error. Yellow box on the gallery is padded 8 px so tiny marks stay visible; area stats use the raw official mask.

In [ ]:
rows = []
for case in report["test_fn"]:
    rows.append(
        {
            "id": case["sample_id"],
            "score": round(case["y_score"], 4),
            "mask_px": case["mask_foreground_pixels"],
            "area": round(case["area_fraction"], 6),
            "bin": case["area_bin"],
            "letterbox_px": case["letterbox_mask_pixels"],
            "location": case["location_bin"],
        }
    )
display(rows)
print("Tiny-dataset table:")
display(report["tiny_defects"])

## Area / location stratification and galleries

In [ ]:
display(report["test_area_stratification"])
display(report["test_location_stratification"])
figures = REPORT_DIR / "figures"
for name in (
    "gallery_fn_test.png",
    "gallery_fp_test.png",
    "area_stratification.png",
    "score_strip.png",
    "gallery_gradcam_fn.png",
    "gallery_gradcam_tp.png",
    "gallery_gradcam_fp.png",
):
    path = figures / name
    if path.is_file():
        display(Markdown(f"**{name}**"))
        display(Image(filename=str(path)))

## Grad-CAM counts

Hit = high-CAM ( ≥ 0.5 × max ) overlaps the official mask. Peak neighborhood is a 16 px dilation.

In [ ]:
display(report["gradcam"]["fn"])
display(report["gradcam"]["tp"])
display(report["gradcam"]["fp"])
display(report["gradcam"]["fp_peak_location"])